# 03 - Modelo Principal: Random Forest
## Proyecto Final TI24 — Bagging: Random Forests
**Estudiante:** Fabio Mavric

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, roc_curve, auc, roc_auc_score)
from sklearn.model_selection import cross_val_score, GridSearchCV
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print('✅ Librerías cargadas')

## 1. Carga de Datos Preprocesados

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test  = pd.read_csv('../data/processed/y_test.csv').values.ravel()

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 2. Fundamentos Teóricos del Random Forest

### ¿Qué es Bagging?
**Bagging** (Bootstrap Aggregating) es una técnica de ensemble que:
1. Genera **B** subconjuntos de datos mediante muestreo con reemplazo (bootstrap)
2. Entrena un modelo en cada subconjunto
3. Combina las predicciones por votación (clasificación) o promedio (regresión)

$$\hat{f}_{bag}(x) = \frac{1}{B} \sum_{b=1}^{B} \hat{f}^{*b}(x)$$

### Random Forest = Bagging + Feature Randomness
En cada split de cada árbol, **solo se evalúan $m$ variables aleatorias** (no todas las $p$ disponibles):

$$m = \sqrt{p} \quad \text{(clasificación)}$$

### Impureza de Gini (criterio de split)
$$G = 1 - \sum_{k=1}^{K} p_k^2$$

donde $p_k$ es la proporción de clase $k$ en el nodo.

### Importancia de Variables
$$\text{Importance}(X_j) = \frac{1}{B} \sum_{b=1}^{B} \sum_{t: \text{split en } X_j} \Delta G_t$$

## 3. Entrenamiento del Modelo

In [ ]:
# Modelo con hiperparámetros elegidos
rf_model = RandomForestClassifier(
    n_estimators=200,        # Número de árboles
    max_depth=12,            # Profundidad máxima de cada árbol
    min_samples_split=10,    # Mínimo de muestras para hacer un split
    min_samples_leaf=5,      # Mínimo de muestras en hoja
    max_features='sqrt',     # m = sqrt(p) — aleatorización de features
    class_weight='balanced', # Manejo de clases desbalanceadas
    random_state=42,
    n_jobs=-1                # Usar todos los núcleos disponibles
)

print('Entrenando Random Forest...')
rf_model.fit(X_train, y_train)
print('✅ Modelo entrenado exitosamente')
print(f'Número de árboles: {rf_model.n_estimators}')
print(f'Features evaluados por split: {rf_model.max_features}')

## 4. Validación Cruzada

In [ ]:
# Cross-validation con 5 folds
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)

print('=== VALIDACIÓN CRUZADA (5-Fold) ===')
print(f'Scores por fold: {[f"{s:.4f}" for s in cv_scores]}')
print(f'Accuracy promedio: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Visualización
plt.figure(figsize=(8, 4))
plt.bar(range(1, 6), cv_scores, color='#3498db', edgecolor='white')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=2,
            label=f'Promedio: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Validación Cruzada — Random Forest (5-Fold)', fontweight='bold')
plt.ylim(0.6, 1.0)
plt.legend()
plt.tight_layout()
plt.savefig('../docs/rf_cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Predicciones y Métricas

In [ ]:
y_pred_rf    = rf_model.predict(X_test)
y_proba_rf   = rf_model.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf  = f1_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_proba_rf)

print('=== MÉTRICAS DEL RANDOM FOREST ===')
print(f'Accuracy:  {acc_rf:.4f} ({acc_rf*100:.2f}%)')
print(f'F1-Score:  {f1_rf:.4f}')
print(f'AUC-ROC:   {auc_rf:.4f}')
print('\n=== REPORTE DE CLASIFICACIÓN ===')
print(classification_report(y_test, y_pred_rf,
                             target_names=['Sin Enfermedad', 'Con Enfermedad']))

## 6. Matriz de Confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sin Enfermedad', 'Con Enfermedad'],
            yticklabels=['Sin Enfermedad', 'Con Enfermedad'],
            linewidths=2, cbar_kws={'shrink': 0.8})
plt.xlabel('Predicción', fontsize=12)
plt.ylabel('Real', fontsize=12)
plt.title('Matriz de Confusión — Random Forest', fontsize=14, fontweight='bold')

# Añadir métricas al margen
total = cm.sum()
plt.figtext(0.02, 0.02, f'Accuracy: {acc_rf*100:.2f}%  |  F1: {f1_rf:.4f}',
            fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('../docs/rf_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Curva ROC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#3498db', linewidth=3, label=f'Random Forest (AUC = {auc_rf:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1.5, label='Clasificador Aleatorio')
plt.fill_between(fpr, tpr, alpha=0.1, color='#3498db')
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
plt.title('Curva ROC — Random Forest', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/rf_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Importancia de Variables

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 6))
colors = ['#e74c3c' if v > importances.mean() else '#3498db' for v in importances.values]
plt.barh(importances.index, importances.values, color=colors, edgecolor='white')
plt.axvline(importances.mean(), color='red', linestyle='--', linewidth=1.5, label='Importancia media')
plt.xlabel('Importancia (Gini)', fontsize=12)
plt.title('Importancia de Variables — Random Forest', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../docs/rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 variables más importantes:')
print(importances.sort_values(ascending=False).head(5).to_string())

## 9. Efecto del Número de Árboles

In [ ]:
n_trees_range = [1, 5, 10, 20, 50, 100, 150, 200]
accs = []

for n in n_trees_range:
    clf = RandomForestClassifier(n_estimators=n, max_depth=12, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    accs.append(accuracy_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.plot(n_trees_range, accs, marker='o', color='#3498db', linewidth=2.5, markersize=8)
plt.xlabel('Número de Árboles (n_estimators)', fontsize=12)
plt.ylabel('Accuracy en Test', fontsize=12)
plt.title('Efecto del Número de Árboles sobre la Accuracy', fontsize=14, fontweight='bold')
plt.xticks(n_trees_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/rf_n_trees_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado — muestra convergencia del error con más árboles')

## 10. Resumen de Resultados

In [ ]:
# Guardar resultados para comparación
import json

rf_results = {
    'model': 'Random Forest',
    'accuracy': float(acc_rf),
    'f1_score': float(f1_rf),
    'auc_roc': float(auc_rf),
    'cv_mean': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std())
}

with open('../data/processed/rf_results.json', 'w') as f:
    json.dump(rf_results, f, indent=2)

print('=' * 50)
print('     RESUMEN FINAL — RANDOM FOREST')
print('=' * 50)
print(f'Accuracy:         {acc_rf*100:.2f}%')
print(f'F1-Score:         {f1_rf:.4f}')
print(f'AUC-ROC:          {auc_rf:.4f}')
print(f'CV Accuracy:      {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
print('=' * 50)
print('✅ Continuar con 04_model_comparison.ipynb')